# 03 - ABMIL baseline training

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/koushikkkkkkkkkk/prostate-mil-project/blob/main/notebooks/03_abmil_training.ipynb)

**Owner:** Person 1 (Team Leader & ABMIL Baseline Owner)

Train the gated-attention ABMIL baseline that CLAM-SB is compared against. Same
split, same features, same seeds as every other model -- that is what makes the
comparison paired and therefore statistically meaningful.


## Setup

In [ ]:
# If running in Google Colab, clone repo and install dependencies:
try:
    import google.colab
    IN_COLAB = True
    import os
    if not os.path.exists("prostate-mil-project"):
        !git clone https://github.com/koushikkkkkkkkkk/prostate-mil-project.git
    %cd prostate-mil-project
    !git pull origin main
    !pip install -r requirements.txt
except ImportError:
    IN_COLAB = False
print("Running in Colab:", IN_COLAB)


In [ ]:
import os, sys, json
from pathlib import Path
os.environ.setdefault("MPLBACKEND", "Agg")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd, torch
print("torch", torch.__version__)

In [ ]:
# Determine dataset: real features vs mock features
has_real = any((ROOT / "data/features").glob("*.pt")) if (ROOT / "data/features").exists() else False

if has_real and (ROOT / "data/common_split.csv").exists():
    MANIFEST = ROOT / "data/common_split.csv"
    FEATURES = ROOT / "data/features"
    CONFIG = ROOT / "configs/abmil.yaml"
    OUT = ROOT / "results/metrics/abmil"
    SYNTHETIC = False
    print("Found REAL features! Using:", MANIFEST)
else:
    # Generate synthetic mock features if not already present
    if not (ROOT / "data/mock/manifest.csv").exists():
        print("No real features in data/features/. Generating synthetic mock dataset for testing...")
        import subprocess
        subprocess.run([sys.executable, "scripts/make_mock_features.py", "--out-dir", "data/mock", "--seed", "7"], cwd=ROOT, check=True)
    MANIFEST = ROOT / "data/mock/manifest.csv"
    FEATURES = ROOT / "data/mock/features"
    CONFIG = ROOT / "configs/abmil_mock.yaml"
    OUT = ROOT / "results/mock_abmil"
    SYNTHETIC = True
    print("Using MOCK dataset for verification:", MANIFEST)

print("Config:", CONFIG)
print("Output dir:", OUT)


## Train (3 seeds on real data, 1 seed on mock)

In [ ]:
import subprocess, sys

# Ensure variables are defined even if previous cells were skipped
if "SYNTHETIC" not in globals():
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    MANIFEST = ROOT / "data/mock/manifest.csv" if (ROOT / "data/mock/manifest.csv").exists() else ROOT / "data/common_split.csv"
    FEATURES = MANIFEST.parent / "features"
    CONFIG = ROOT / ("configs/abmil_mock.yaml" if "mock" in str(MANIFEST) else "configs/abmil.yaml")
    OUT = ROOT / ("results/mock_abmil" if "mock" in str(MANIFEST) else "results/metrics/abmil")
    SYNTHETIC = "mock" in str(MANIFEST)

seeds = ["0"] if SYNTHETIC else ["0", "1", "2"]
cmd = [sys.executable, "-m", "src.train", "--model", "abmil",
       "--manifest", str(MANIFEST), "--features-root", str(FEATURES),
       "--config", str(CONFIG), "--out-dir", str(OUT), "--seeds", *seeds]
print("Running:", " ".join(cmd))
res = subprocess.run(cmd, cwd=ROOT, capture_output=True, text=True)
if res.stdout:
    print(res.stdout[-2500:])
if res.stderr:
    print("STDERR:", res.stderr[-1000:])
if res.returncode != 0:
    raise RuntimeError(f"Training exited with code {res.returncode}")
print("\n✓ Training finished successfully!")


## Results and confusion matrix

In [ ]:
if "OUT" not in globals():
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    OUT = ROOT / "results/mock_abmil" if (ROOT / "results/mock_abmil").exists() else ROOT / "results/metrics/abmil"

aggregate_file = OUT / "aggregate_abmil.json"
if aggregate_file.exists():
    aggregate = json.loads(aggregate_file.read_text())
    table_file = OUT / "table_abmil.md"
    if table_file.exists():
        print(table_file.read_text())
    display(pd.DataFrame(aggregate.get("per_run", [])))
else:
    print(f"No results found in {OUT}. Run the training cell above first.")


In [ ]:
from IPython.display import Image, display
if "OUT" not in globals():
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    OUT = ROOT / "results/mock_abmil" if (ROOT / "results/mock_abmil").exists() else ROOT / "results/metrics/abmil"

pngs = sorted(OUT.glob("confusion_*.png"))
if pngs:
    display(Image(filename=str(pngs[0]), width=450))
else:
    print(f"No confusion matrix found in {OUT}.")


## What Person 2 needs from this notebook

* `results/metrics/abmil/predictions_abmil_seed0.csv` -- the prediction file used
  in the paired CLAM-SB vs ABMIL comparison.
* The aggregate QWK, so the comparison can quote both numbers side by side.